# Paper Evaluation Pipeline (Notebook)

与 `run_paper_eval.py` 等价，可在线上容器中按单元格运行。

**使用前请修改下方「配置」单元格中的 `exp_id`、`output_dir` 等。**

若本机 PyTorch 报 HIP/libgalaxyhip 错误，可在仅安装 CPU 版 PyTorch 的容器中运行，并在配置里设置 `use_cpu = True`。

## 1. 路径与配置

In [1]:
import os
import sys

# 若在 Jupyter 中运行，通常 cwd 为 vis_mlp 或 paper_eval
try:
    _file = __file__
except NameError:
    _file = os.path.join(os.getcwd(), "paper_eval", "run_paper_eval.ipynb")
BASE = os.path.dirname(os.path.dirname(os.path.abspath(_file))) if "ipynb" not in _file else os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(BASE, "paper_eval")):
    BASE = os.path.dirname(BASE)
PAPER_EVAL_DIR = os.path.join(BASE, "paper_eval")
sys.path.insert(0, BASE)
sys.path.insert(0, os.path.join(BASE, "train"))
os.chdir(BASE)

# ---------- 配置（按需修改） ----------
exp_id = "exp_1772692533"
exp_id = "exp_1773291129_old3"
output_dir = None  # None 则用 paper_eval_results_{exp_id}
data_dir = os.path.join(BASE, "ml_dataset_cursor_2_12h")
ckpt_dir = os.path.join(BASE, "checkpoints")
shp_path = "/public/home/putianshu/中华人民共和国/中华人民共和国.shp"
ifs_nc = os.path.join(BASE, "VIS_IDW_KDTree_20250101_20251231.nc")
fog_th = 0.10
mist_th = 0.38
# exp_1773291129_old2: train uses Fog:0.10,Mist:0.38. With fog_th=0.46 Mist recall drops to ~0. Use no_calibration=False or fog_th=0.10.

no_calibration = True  # False: load season_thresholds.pt + temperature; True: use global fog_th/mist_th above
# 阈值规则: "default"(Fog覆盖Mist), "mutual"(与训练一致:双超阈值时概率高者胜,减轻Mist->Fog), "joint"(归一化超出度)
threshold_rule = "mutual"
use_cpu = False  # 容器实例中无 GPU 或 HIP 报错时设为 True
batch_size = 8192  # 推理 batch，多 DCU 可设大一些加速（CPU 建议 1024）

# 大范围雾事件参数
event_top_k = 3
event_window_hours = 3
event_min_fog_stations = 80
event_min_regions = 3
event_min_lon_span = 10.0
event_min_lat_span = 4.0
event_gap_hours = 24

output_dir = output_dir or os.path.join(BASE, f"paper_eval_results_{exp_id}")
os.makedirs(output_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, f"{exp_id}_S2_PhaseB_best_score.pt")
scaler_path = os.path.join(ckpt_dir, "robust_scaler_w12.pkl")
season_th_path = os.path.join(ckpt_dir, f"{exp_id}_season_thresholds.pt")

print("BASE:", BASE)
print("Output:", output_dir)
print("IFS:", ifs_nc)
print("use_cpu:", use_cpu)

BASE: /public/home/putianshu/vis_mlp
Output: /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3
IFS: /public/home/putianshu/vis_mlp/VIS_IDW_KDTree_20250101_20251231.nc
use_cpu: False


## 2. 导入（不含 PyTorch）

In [2]:
import os
import sys
import types
import joblib
import numpy as np
import pandas as pd
import importlib.util
from sklearn.metrics import classification_report

sys.dont_write_bytecode = True
importlib.invalidate_caches()

# 清理旧的包/子模块缓存，避免 notebook 长生命周期内混用新旧模块
for _name in list(sys.modules.keys()):
    if _name == "paper_eval" or _name.startswith("paper_eval."):
        del sys.modules[_name]

# 构造一个指向源码目录的轻量 package，使相对导入 `.plot_style` / `.metrics_core` 可正常工作
paper_eval_pkg = types.ModuleType("paper_eval")
paper_eval_pkg.__path__ = [PAPER_EVAL_DIR]
paper_eval_pkg.__file__ = os.path.join(PAPER_EVAL_DIR, "__init__.py")
sys.modules["paper_eval"] = paper_eval_pkg


def _load_paper_eval_submodule(submodule_name):
    full_name = f"paper_eval.{submodule_name}"
    module_path = os.path.join(PAPER_EVAL_DIR, f"{submodule_name}.py")
    spec = importlib.util.spec_from_file_location(full_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = module
    spec.loader.exec_module(module)
    return module

# 直接从源文件加载，但保留 package 语义，彻底绕开旧 pyc / 包缓存
plot_style_mod = _load_paper_eval_submodule("plot_style")
metrics_core_mod = _load_paper_eval_submodule("metrics_core")
plot_classification_mod = _load_paper_eval_submodule("plot_classification")
plot_spatial_mod = _load_paper_eval_submodule("plot_spatial")
plot_scenarios_mod = _load_paper_eval_submodule("plot_scenarios")

pred_from_thresholds = metrics_core_mod.pred_from_thresholds
pred_from_thresholds_mutual = getattr(metrics_core_mod, "pred_from_thresholds_mutual", pred_from_thresholds)
pred_from_joint_thresholds = getattr(metrics_core_mod, "pred_from_joint_thresholds", pred_from_thresholds)
compute_rare_event_report = metrics_core_mod.compute_rare_event_report
if hasattr(metrics_core_mod, "pred_from_season_thresholds"):
    pred_from_season_thresholds = metrics_core_mod.pred_from_season_thresholds
else:
    def pred_from_season_thresholds(probs, months, season_thresholds, default_fog_th=0.46, default_mist_th=0.38, rule="default"):
        month_to_season = {
            12: "DJF", 1: "DJF", 2: "DJF",
            3: "MAM", 4: "MAM", 5: "MAM",
            6: "JJA", 7: "JJA", 8: "JJA",
            9: "SON", 10: "SON", 11: "SON",
        }
        months = np.asarray(months, dtype=np.int32).ravel()
        n = probs.shape[0]
        fog_th_arr = np.full(n, default_fog_th, dtype=np.float64)
        mist_th_arr = np.full(n, default_mist_th, dtype=np.float64)
        for i in range(n):
            season = month_to_season.get(int(months[i]))
            if season and season in season_thresholds:
                fog_th_arr[i] = season_thresholds[season]["fog_th"]
                mist_th_arr[i] = season_thresholds[season]["mist_th"]
        if rule == "mutual":
            return pred_from_thresholds_mutual(probs, fog_th_arr, mist_th_arr)
        if rule == "joint":
            return pred_from_joint_thresholds(probs, fog_th_arr, mist_th_arr)
        return metrics_core_mod.pred_from_thresholds(probs, fog_th_arr, mist_th_arr)

setup_paper_style = plot_style_mod.setup_paper_style
save_figure = plot_style_mod.save_figure
plot_confusion_matrix_normalized = plot_classification_mod.plot_confusion_matrix_normalized
plot_per_class_prf1 = plot_classification_mod.plot_per_class_prf1
plot_pr_curves = plot_classification_mod.plot_pr_curves
plot_threshold_sweep = plot_classification_mod.plot_threshold_sweep
plot_reliability_diagram = plot_classification_mod.plot_reliability_diagram
plot_fog_recall_map = plot_spatial_mod.plot_fog_recall_map
plot_fpr_map = plot_spatial_mod.plot_fpr_map
run_widespread_event_evaluation = getattr(plot_spatial_mod, "run_widespread_event_evaluation", None)
plot_scenario_bars = plot_scenarios_mod.plot_scenario_bars

if hasattr(plot_scenarios_mod, "derive_scenario_columns"):
    derive_scenario_columns = plot_scenarios_mod.derive_scenario_columns
else:
    def derive_scenario_columns(meta):
        df = meta.copy()
        if "latitude" in df.columns and "lat" not in df.columns:
            df["lat"] = df["latitude"]
        if "longitude" in df.columns and "lon" not in df.columns:
            df["lon"] = df["longitude"]
        if df["time"].dtype == object or "datetime" not in str(df["time"].dtype):
            df["time"] = pd.to_datetime(df["time"])
        df["hour"] = df["time"].dt.hour.values
        df["month"] = df["time"].dt.month.values
        lats = df["lat"].values
        lons = df["lon"].values
        df["is_coastal"] = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(int)
        region = np.full(len(df), "Other", dtype=object)
        region_defs = [
            ("Northeast", 38.5, 54, 118, 136),
            ("North_China", 32, 42.5, 110, 120),
            ("East_China", 27, 35, 115, 123),
            ("Central_China", 26, 34, 108, 118),
            ("South_China", 18, 26, 105, 120),
            ("Southwest", 21, 35, 97, 108),
            ("Northwest", 31, 50, 73, 111),
        ]
        for name, lat_min, lat_max, lon_min, lon_max in region_defs:
            mask = (lats >= lat_min) & (lats <= lat_max) & (lons >= lon_min) & (lons <= lon_max)
            region[mask] = name
        df["region"] = region
        return df

if hasattr(plot_scenarios_mod, "build_confusion_summaries_and_bottleneck_table"):
    build_confusion_summaries_and_bottleneck_table = plot_scenarios_mod.build_confusion_summaries_and_bottleneck_table
else:
    def build_confusion_summaries_and_bottleneck_table(eval_df, output_dir):
        def _confusion_type(y_true, pred):
            if y_true == pred:
                return "correct"
            if y_true == 0 and pred == 1:
                return "fog->mist"
            if y_true == 1 and pred == 0:
                return "mist->fog"
            if y_true == 1 and pred == 2:
                return "mist->clear"
            if y_true == 2 and pred == 1:
                return "clear->mist"
            if y_true == 2 and pred == 0:
                return "clear->fog"
            return "other"

        df = eval_df.copy()
        df["confusion_type"] = [
            _confusion_type(int(df["y_true"].iloc[i]), int(df["pred"].iloc[i]))
            for i in range(len(df))
        ]
        season_map = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
        df["season"] = df["month"].map(season_map)
        df["day_night"] = df["hour"].map(lambda h: "day" if 6 <= int(h) < 18 else "night")
        if "is_coastal" in df.columns:
            df["coastal_inland"] = df["is_coastal"].map(lambda x: "coastal" if x == 1 else "inland")

        confusion_types = ["fog->mist", "mist->fog", "mist->clear", "clear->mist", "clear->fog"]
        strata = ["season", "day_night", "coastal_inland", "region", "visibility_band"]
        rows_summary = []
        for col in strata:
            if col not in df.columns:
                continue
            for slice_val in df[col].dropna().unique():
                sub = df[df[col] == slice_val]
                n_s = len(sub)
                for ct in confusion_types:
                    cnt = int((sub["confusion_type"] == ct).sum())
                    n_err_s = int((sub["confusion_type"] != "correct").sum())
                    rows_summary.append({
                        "slice_dim": col,
                        "slice_value": slice_val,
                        "confusion_type": ct,
                        "count": cnt,
                        "n_slice": n_s,
                        "rate_over_slice": round(cnt / n_s, 6) if n_s > 0 else 0.0,
                        "rate_among_errors_slice": round(cnt / n_err_s, 6) if n_err_s > 0 else 0.0,
                    })
        pd.DataFrame(rows_summary).to_csv(os.path.join(output_dir, "confusion_summaries_stratified.csv"), index=False)

        n_total = len(df)
        jja_mask = df["season"] == "JJA"
        boundary_bands = ["400-600", "600-800", "800-1200"]
        bnd_mask = df["visibility_band"].isin(boundary_bands) if "visibility_band" in df.columns else np.zeros(n_total, dtype=bool)
        rows = []
        for ct in confusion_types:
            full_count = int((df["confusion_type"] == ct).sum())
            jja_count = int((df.loc[jja_mask, "confusion_type"] == ct).sum())
            bnd_count = int((df.loc[bnd_mask, "confusion_type"] == ct).sum())
            rows.extend([
                {"confusion_type": ct, "scope": "full", "count": full_count, "rate": round(full_count / n_total, 6) if n_total > 0 else 0.0, "n_scope": n_total},
                {"confusion_type": ct, "scope": "JJA", "count": jja_count, "rate": round(jja_count / int(jja_mask.sum()), 6) if int(jja_mask.sum()) > 0 else 0.0, "n_scope": int(jja_mask.sum())},
                {"confusion_type": ct, "scope": "boundary_zone", "count": bnd_count, "rate": round(bnd_count / int(bnd_mask.sum()), 6) if int(bnd_mask.sum()) > 0 else 0.0, "n_scope": int(bnd_mask.sum())},
            ])
        bottleneck_df = pd.DataFrame(rows)
        full_rank = bottleneck_df[bottleneck_df["scope"] == "full"].sort_values("count", ascending=False)
        order = full_rank["confusion_type"].tolist()
        bottleneck_df["rank_full_count"] = bottleneck_df["confusion_type"].map({c: i + 1 for i, c in enumerate(order)})
        bottleneck_df.to_csv(os.path.join(output_dir, "bottleneck_table.csv"), index=False)

if run_widespread_event_evaluation is None:
    print("Note: run_widespread_event_evaluation not found in plot_spatial (event evaluation will be skipped).")

print("Non-torch imports OK.")

Non-torch imports OK.


## 3. 辅助函数

In [3]:
def load_test_data(data_dir, scaler_path, window_size=12):
    X_path = os.path.join(data_dir, "X_test.npy")
    y_path = os.path.join(data_dir, "y_test.npy")
    meta_path = os.path.join(data_dir, "meta_test.csv")
    for p in [X_path, y_path, meta_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Not found: {p}")
    y_raw = np.load(y_path)
    if np.max(y_raw) < 100:
        y_raw = y_raw * 1000.0
    y_cls = np.zeros(len(y_raw), dtype=np.int64)
    y_cls[y_raw >= 500] = 1
    y_cls[y_raw >= 1000] = 2
    meta = pd.read_csv(meta_path, parse_dates=["time"])
    meta["hour"] = meta["time"].dt.hour
    meta["month"] = meta["time"].dt.month
    scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None
    return X_path, y_cls, y_raw, meta, scaler


# 与训练脚本 CONFIG 一致，否则模型与 checkpoint 维度不匹配
EXTRA_FEAT_DIMS = 36   # FE_EXTRA_DIMS
REGIME_FEAT_DIM = 6    # REGIME_FEAT_DIM

def _regime_features_from_meta(meta_slice):
    """从 meta 的 month/hour/lat,lon 得到 6 维 regime，与训练脚本一致。"""
    month = np.asarray(meta_slice["month"].values, dtype=np.int32)
    hour = np.asarray(meta_slice["hour"].values, dtype=np.int32)
    lats = meta_slice["lat"].values if "lat" in meta_slice.columns else np.zeros(len(month))
    lons = meta_slice["lon"].values if "lon" in meta_slice.columns else np.zeros(len(month))
    is_coastal = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(np.float32)
    is_day = ((hour >= 6) & (hour < 18)).astype(np.float32)
    djf = ((month == 12) | (month <= 2)).astype(np.float32)
    mam = ((month >= 3) & (month <= 5)).astype(np.float32)
    jja = ((month >= 6) & (month <= 8)).astype(np.float32)
    son = ((month >= 9) & (month <= 11)).astype(np.float32)
    return np.column_stack([djf, mam, jja, son, is_day, is_coastal]).astype(np.float32)

def run_inference(X_path, scaler, model, device, batch_size=1024, window_size=12, temperature=None, meta=None):
    import torch
    import torch.nn.functional as F
    T = 1.0 if temperature is None or temperature == 1.0 else float(temperature)
    X = np.load(X_path, mmap_mode="r")
    N = len(X)
    split_dyn = 25 * window_size
    log_mask = np.zeros(split_dyn, dtype=bool)
    for t in range(window_size):
        for i in [2, 4, 9]:
            log_mask[t * 25 + i] = True
    use_cuda = device.type == "cuda"
    non_blocking = use_cuda
    need_regime = getattr(model, "regime_feat_dim", 0) if not hasattr(model, "module") else getattr(model.module, "regime_feat_dim", 0)
    if need_regime and meta is None:
        raise ValueError("model has regime_feat_dim but meta was not passed to run_inference")
    all_probs = []
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        rows = X[start:end].astype(np.float32)
        feats = rows[:, : split_dyn + 5]
        feats[:, :split_dyn] = np.where(
            log_mask,
            np.log1p(np.maximum(feats[:, :split_dyn], 0)),
            feats[:, :split_dyn],
        )
        if scaler is not None:
            feats = (feats - scaler.center_) / (scaler.scale_ + 1e-6)
        veg = rows[:, split_dyn + 5 : split_dyn + 6]
        extra = rows[:, split_dyn + 6 :]
        if extra.shape[1] < EXTRA_FEAT_DIMS:
            extra = np.pad(extra, ((0, 0), (0, EXTRA_FEAT_DIMS - extra.shape[1])), mode="constant", constant_values=0)
        elif extra.shape[1] > EXTRA_FEAT_DIMS:
            extra = extra[:, :EXTRA_FEAT_DIMS]
        final = np.concatenate([np.clip(feats, -10, 10), veg, np.clip(extra, -10, 10)], axis=1)
        if need_regime and meta is not None:
            regime = _regime_features_from_meta(meta.iloc[start:end])
            final = np.concatenate([final, regime], axis=1)
        final = np.nan_to_num(final, nan=0.0)
        x = torch.from_numpy(final).float().to(device, non_blocking=non_blocking)
        with torch.inference_mode():
            fine, _, _ = model(x)
            probs = F.softmax(fine / T, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def visibility_boundary_band(y_raw):
    """能见度区间标签: <400, 400-600, 600-800, 800-1200, >1200 (米)"""
    y_raw = np.asarray(y_raw, dtype=np.float64)
    band = np.full(len(y_raw), ">1200", dtype=object)
    band[y_raw < 400] = "<400"
    band[(y_raw >= 400) & (y_raw < 600)] = "400-600"
    band[(y_raw >= 600) & (y_raw < 800)] = "600-800"
    band[(y_raw >= 800) & (y_raw < 1200)] = "800-1200"
    return band


def export_per_sample_table(meta, y_true, y_raw, pred, probs, output_path):
    """导出逐样本表: y_true, y_raw, pred, p_fog, p_mist, month, hour, station_id, lat, lon, visibility_band"""
    df = meta[["station_id", "lat", "lon", "time", "month", "hour"]].copy()
    df["y_true"] = y_true
    df["y_raw"] = y_raw
    df["pred"] = pred
    df["p_fog"] = probs[:, 0]
    df["p_mist"] = probs[:, 1]
    df["visibility_band"] = visibility_boundary_band(y_raw)
    df.to_csv(output_path, index=False, float_format="%.6f")
    return df


def aggregate_station_metrics(meta, y_true, preds):
    stations = meta[["station_id", "lat", "lon"]].drop_duplicates("station_id")
    rows = []
    for _, row in stations.iterrows():
        sid = row["station_id"]
        m = (meta["station_id"] == sid).values
        y_s = y_true[m]
        p_s = preds[m]
        n_fog_true = (y_s == 0).sum()
        n_mist_true = (y_s == 1).sum()
        n_clear_true = (y_s == 2).sum()
        fog_recall = ((y_s == 0) & (p_s == 0)).sum() / n_fog_true if n_fog_true > 0 else 0.0
        fog_pred = (p_s == 0)
        fog_precision = ((y_s == 0) & fog_pred).sum() / fog_pred.sum() if fog_pred.sum() > 0 else 0.0
        mist_recall = ((y_s == 1) & (p_s == 1)).sum() / n_mist_true if n_mist_true > 0 else 0.0
        mist_pred = (p_s == 1)
        mist_precision = ((y_s == 1) & mist_pred).sum() / mist_pred.sum() if mist_pred.sum() > 0 else 0.0
        fpr_val = ((p_s <= 1) & (y_s == 2)).sum() / n_clear_true if n_clear_true > 0 else 0.0
        rows.append({
            "station_id": sid, "lat": row["lat"], "lon": row["lon"],
            "fog_recall": fog_recall, "fog_precision": fog_precision,
            "mist_recall": mist_recall, "mist_precision": mist_precision,
            "fpr_fog": fpr_val, "overall_acc": (p_s == y_s).mean(),
            "n_fog": n_fog_true, "n_mist": n_mist_true, "n_clear": n_clear_true, "n_total": len(y_s),
        })
    return pd.DataFrame(rows)

print("Helper functions defined.")

Helper functions defined.


## 4. 加载 PyTorch 与模型

In [4]:
import torch
import torch.nn.functional as F

if use_cpu or os.environ.get("CUDA_VISIBLE_DEVICES") == "":
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# exp_1773291129_old2 was trained with PMST_net_test_11_s2 (temporal 13, extra 36). Use 11_s2 for that ckpt.
if "old2" in exp_id:
    from PMST_net_test_11_s2 import ImprovedDualStreamPMSTNet
    _extra_dim = 36
else:
    from PMST_net_test_10_s2 import ImprovedDualStreamPMSTNet
    _extra_dim = 36

model = ImprovedDualStreamPMSTNet(
    window_size=12, hidden_dim=512, dropout=0.3, extra_feat_dim=_extra_dim
).to(device)
# 确认当前导入的模型维度（应与 checkpoint 一致，否则 load_state_dict 会报错）
print("导入模块:", ImprovedDualStreamPMSTNet.__module__)
print("temporal 输入维度:", model.temporal_input_proj.in_features, "(temporal_input_proj.weight 应为 [512, 此数])")
state = torch.load(ckpt_path, map_location=device)
state = {k.replace("module.", ""): v for k, v in state.items()}
if "temporal_input_proj.weight" in state:
    ckpt_in = state["temporal_input_proj.weight"].shape[1]
    print("checkpoint 中 temporal 输入维度:", ckpt_in)
    if ckpt_in != model.temporal_input_proj.in_features:
        print("  [错误] 与当前模型维度不一致，请改用训练该 checkpoint 时使用的脚本（如 PMST_net_test_11_s2）或检查导入路径。")
model.load_state_dict(state, strict=False)
model.eval()
if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print("Model loaded (DataParallel on", torch.cuda.device_count(), "devices):", ckpt_path)
else:
    print("Device:", device)
    print("Model loaded:", ckpt_path)
if batch_size is None or (use_cpu and batch_size > 1024):
    batch_size = 1024 if use_cpu else 8192
print("Inference batch_size:", batch_size)

# 加载校准 artifact（季节阈值 + 温度），默认启用
season_thresholds = None
temperature = None
if not no_calibration and os.path.exists(season_th_path):
    try:
        try:
            cal = torch.load(season_th_path, map_location="cpu", weights_only=True)
        except TypeError:
            cal = torch.load(season_th_path, map_location="cpu")
        season_thresholds = cal.get("season_thresholds") or None
        temperature = cal.get("temperature")
        if temperature is not None:
            temperature = float(temperature)
        print("Calibration loaded:", season_th_path, "| temperature:", temperature, "| seasons:", list(season_thresholds.keys()) if season_thresholds else [])
    except Exception as e:
        print("WARN: Could not load calibration:", e)
if season_thresholds is None:
    print("Using global thresholds (no artifact or no_calibration=True).")

导入模块: PMST_net_test_10_s2
temporal 输入维度: 13 (temporal_input_proj.weight 应为 [512, 此数])
checkpoint 中 temporal 输入维度: 13
Model loaded (DataParallel on 4 devices): /public/home/putianshu/vis_mlp/checkpoints/exp_1773291129_old3_S2_PhaseB_best_score.pt
Inference batch_size: 8192
Using global thresholds (no artifact or no_calibration=True).


## 5. 加载数据

In [5]:
X_path, y_cls, y_raw, meta, scaler = load_test_data(data_dir, scaler_path, window_size=12)
print(f"Samples: {len(y_cls)}, Stations: {meta['station_id'].nunique()}")

Samples: 1166712, Stations: 2167


## 6. 推理与阈值预测

In [6]:
probs = run_inference(X_path, scaler, model, device, batch_size=batch_size, window_size=12, temperature=temperature)
# threshold_rule 在配置单元格定义: "default" | "mutual"(与训练一致,减轻 Mist->Fog) | "joint"
rule = threshold_rule if "threshold_rule" in dir() else "default"
if season_thresholds is not None:
    preds = pred_from_season_thresholds(probs, meta["month"].values, season_thresholds, fog_th, mist_th, rule=rule)
else:
    if rule == "mutual":
        preds = pred_from_thresholds_mutual(probs, fog_th, mist_th)
    elif rule == "joint":
        preds = pred_from_joint_thresholds(probs, fog_th, mist_th)
    else:
        preds = pred_from_thresholds(probs, fog_th, mist_th)
report = compute_rare_event_report(probs, y_cls, fog_th, mist_th, pred=preds)

print("Rare-event metrics:")
for k, v in report.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")

Rare-event metrics:
  Fog_P: 0.2342
  Fog_R: 0.5022
  Fog_F1: 0.3194
  Mist_P: 0.0815
  Mist_R: 0.3549
  Mist_F1: 0.1326
  Clear_P: 0.9922
  Clear_R: 0.9459
  Clear_F1: 0.9685
  Fog_CSI: 0.1901
  Fog_POD: 0.5022
  Fog_FAR: 0.7658
  Fog_PSS: -0.2636
  Fog_HSS: 0.3067
  Mist_CSI: 0.0710
  Mist_POD: 0.3549
  Mist_FAR: 0.9185
  Mist_PSS: -0.5635
  Mist_HSS: 0.1197
  low_vis_precision: 0.2237
  false_positive_rate: 0.0541
  macro_f1: 0.4735
  balanced_acc: 0.6010
  mcc: 17.1541
  Brier_Fog: 0.0156
  Brier_Mist: 0.0220
  ECE: 0.0113


## 7. 保存报告与站级指标

In [7]:
with open(os.path.join(output_dir, "rare_event_report.txt"), "w") as f:
    f.write(classification_report(y_cls, preds, target_names=["Fog", "Mist", "Clear"]))
    f.write("\n\nRare-event metrics:\n")
    for k, v in report.items():
        if isinstance(v, (int, float)):
            f.write(f"  {k}: {v:.4f}\n")

sta_df = aggregate_station_metrics(meta, y_cls, preds)
sta_df.to_csv(os.path.join(output_dir, "station_metrics.csv"), index=False)
print(f"Report and station_metrics.csv saved ({len(sta_df)} stations).")

# 逐样本表与混淆/瓶颈诊断（Priority 1）
per_sample_path = os.path.join(output_dir, "per_sample_eval.csv")
eval_df = export_per_sample_table(meta, y_cls, y_raw, preds, probs, per_sample_path)
print(f"Per-sample table saved: {per_sample_path} ({len(eval_df)} rows).")
eval_df = derive_scenario_columns(eval_df)
build_confusion_summaries_and_bottleneck_table(eval_df, output_dir)

Report and station_metrics.csv saved (2167 stations).
Per-sample table saved: /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/per_sample_eval.csv (1166712 rows).
  [Diagnostics] Stratified confusion summaries -> /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/confusion_summaries_stratified.csv
  [Diagnostics] Bottleneck table (JJA + boundary-zone) -> /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/bottleneck_table.csv


## 8. 生成图表

In [8]:
setup_paper_style()
class_names = ["Fog", "Mist", "Clear"]

plot_confusion_matrix_normalized(
    y_cls, preds, class_names,
    os.path.join(output_dir, "fig3_confusion_matrix.png"),
)
plot_per_class_prf1(report, os.path.join(output_dir, "fig3_prf1_bars.png"))
plot_pr_curves(probs, y_cls, class_names, os.path.join(output_dir, "fig4_pr_curves.png"))
plot_threshold_sweep(probs, y_cls, os.path.join(output_dir, "fig4_threshold_sweep.png"), fog_th, mist_th)
plot_reliability_diagram(probs, y_cls, os.path.join(output_dir, "fig5_reliability.png"))

if os.path.exists(shp_path):
    plot_fog_recall_map(sta_df, os.path.join(output_dir, "fig8_station_fog_recall.png"), shp_path=shp_path, min_fog_events=5)
    plot_fpr_map(sta_df, os.path.join(output_dir, "fig8_station_fpr.png"), shp_path=shp_path, min_clear_events=20)
else:
    print("Shapefile not found, skipping maps.")

meta_full = meta.copy()
meta_full["y_true"] = y_cls
meta_full["pred"] = preds
meta_full["p_fog"] = probs[:, 0]
meta_full["p_mist"] = probs[:, 1]
plot_scenario_bars(meta_full, os.path.join(output_dir, "fig7_scenario_robustness.png"), fog_th=fog_th, mist_th=mist_th)

if run_widespread_event_evaluation is not None:
    try:
        event_df = run_widespread_event_evaluation(
            meta=meta,
            y_true=y_cls,
            y_true_raw=y_raw,
            pmst_pred=preds,
            output_dir=output_dir,
            shp_path=shp_path,
            ifs_nc_path=ifs_nc,
            top_k=event_top_k,
            window_hours=event_window_hours,
            min_fog_stations=event_min_fog_stations,
            min_regions=event_min_regions,
            min_lon_span=event_min_lon_span,
            min_lat_span=event_min_lat_span,
            gap_hours=event_gap_hours,
        )
        print(f"Widespread fog-event evaluation saved for {len(event_df)} events.")
    except Exception as e:
        print(f"[WARN] Event evaluation skipped: {e}")
else:
    print("Skipping widespread event evaluation (run_widespread_event_evaluation not available).")

print("All figures saved to", output_dir)

  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig3_confusion_matrix.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig3_prf1_bars.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig4_pr_curves.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig4_threshold_sweep.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig5_reliability.png
  [Map] Masked 1281 stations with n_fog<5
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig8_station_fog_recall.png
  [Map] Masked 1 stations with n_clear<20
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig8_station_fpr.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig7_scenario_robustness.png
  [Fig] Saved → /public/home/putianshu/v

In [9]:
# Overwrite fig3 with matched PMST-vs-IFS comparison and add Mist map
load_ifs_baseline = plot_spatial_mod.load_ifs_baseline
plot_mist_recall_map = plot_spatial_mod.plot_mist_recall_map

if "report_ifs" not in locals() or "report_matched" not in locals() or "ifs_valid" not in locals():
    ifs_pred = np.full(len(y_cls), -1, dtype=np.int64)
    ifs_vis_raw = np.full(len(y_cls), np.nan, dtype=np.float64)
    ifs_valid = np.zeros(len(y_cls), dtype=bool)
    report_matched = None
    report_ifs = None
    try:
        ifs_pred, ifs_vis_raw, ifs_valid = load_ifs_baseline(meta, ifs_nc)
        n_ifs_valid = int(ifs_valid.sum())
        if n_ifs_valid > 0:
            report_matched = compute_rare_event_report(
                probs[ifs_valid], y_cls[ifs_valid], fog_th, mist_th, pred=preds[ifs_valid]
            )
            report_ifs = compute_rare_event_report(
                np.zeros((n_ifs_valid, probs.shape[1]), dtype=np.float64),
                y_cls[ifs_valid],
                fog_th,
                mist_th,
                pred=ifs_pred[ifs_valid],
            )
            print(f"IFS baseline matched: {n_ifs_valid}/{len(ifs_valid)} samples.")
        else:
            print("[WARN] IFS baseline matched 0 samples; comparison figures will fall back to PMST-only.")
    except Exception as e:
        print(f"[WARN] IFS baseline load skipped: {e}")

setup_paper_style()
class_names = ["Fog", "Mist", "Clear"]

if report_ifs is not None:
    plot_confusion_matrix_normalized(
        y_cls[ifs_valid], preds[ifs_valid], class_names,
        os.path.join(output_dir, "fig3_confusion_matrix.png"),
        baseline_pred=ifs_pred[ifs_valid],
    )
    plot_per_class_prf1(
        report_matched,
        os.path.join(output_dir, "fig3_prf1_bars.png"),
        baseline_stats=report_ifs,
    )
else:
    print("[WARN] IFS comparison unavailable; keeping PMST-only fig3 outputs.")

if os.path.exists(shp_path):
    plot_mist_recall_map(
        sta_df,
        os.path.join(output_dir, "fig8_station_mist_recall.png"),
        shp_path=shp_path,
        min_mist_events=5,
    )
else:
    print("Shapefile not found, skipping mist recall map.")

print("Updated fig3 comparison outputs and fig8_station_mist_recall.png in", output_dir)


  [IFS] Matched 1077001/1166712 samples from VIS_IDW_KDTree_20250101_20251231.nc
IFS baseline matched: 1077001/1166712 samples.
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig3_confusion_matrix.png
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig3_prf1_bars.png
  [Map] Masked 1396 stations with n_mist<5
  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3/fig8_station_mist_recall.png
Updated fig3 comparison outputs and fig8_station_mist_recall.png in /public/home/putianshu/vis_mlp/paper_eval_results_exp_1773291129_old3
